# TorchLens + Hugging Face: tracing, activations, visualization

This notebook shows how to use TorchLens with Hugging Face transformer models — capture a full forward pass, browse the structure, grab activations from any point in the network, and visualize the computation graph. No special model preparation required: TorchLens traces any eager PyTorch model, and HF models are just eager PyTorch under the hood.

The convenience entry point is `torchlens.bridge.hf.trace_text(model, text)` — it auto-resolves the right tokenizer and runs the trace in one call. You can also call `tl.trace(model, input_args=(...))` directly if you've already tokenized.

**What's covered**
1. Load a HF model (DistilBERT — small, fast, runs on CPU)
2. Trace a forward pass and inspect the result
3. Browse layers, ops, modules
4. Grab activations by layer label, by module path, and by function family
5. Visualize the full graph
6. Module-focused subgraphs (zoom into one attention block)
7. Try a different architecture (GPT-2 small) to confirm it generalizes

Requirements: `torchlens`, `transformers`, `graphviz` (system package; Debian/Ubuntu: `apt install graphviz`).

In [ ]:
from __future__ import annotations

from pathlib import Path
import tempfile

import torch
from IPython.display import IFrame, display

import torchlens as tl

# Where we'll save rendered graphs
OUT = Path(tempfile.mkdtemp(prefix="torchlens_hf_demo_"))
print(f"torchlens {tl.__version__}")
print(f"output dir: {OUT}")

## 1. Load a Hugging Face model

We'll use DistilBERT base (uncased) — small enough to run on CPU comfortably, big enough to show off the full pipeline. Any `AutoModel` works the same way.

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("distilbert-base-uncased")
model.eval()
print(f"{type(model).__name__}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

## 2. Trace a forward pass

`tl.bridge.hf.trace_text(model, text)` does three things for you:

1. Resolves the matching tokenizer from `model.name_or_path`
2. Tokenizes the text and packages it as model inputs
3. Runs `tl.trace(model, ...)` with those inputs

The returned `Trace` object holds everything: ops, layers, modules, activations, source code locations, FLOPs, memory, timing.

In [ ]:
log = tl.bridge.hf.trace_text(model, "TorchLens makes transformer internals queryable.")
print(f"model class: {log.model_class_name}")
print(f"num ops:     {log.num_ops}")
print(f"num layers:  {len(log.layers)}")
print(f"num modules: {log.num_modules}")

You can also trace with explicit tokenized inputs if you prefer — useful when you have a tokenizer pipeline already, or want to control batching:

```python
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
inputs = tok("Hello world!", return_tensors='pt')
log = tl.trace(model, input_kwargs=inputs)
```

Both produce equivalent traces. The `trace_text` helper just removes one step.

## 3. Inspect the trace

The trace's `summary()` gives a quick top-level overview.

In [ ]:
print(log.summary())

## 4. Browse the captured layers

Each `Layer` is a graph position; each `Op` is one executed torch function call at that position. For DistilBERT all layers are single-op (no loops), so layers and ops correspond 1:1.

`log.layer_labels` gives every label in execution order. The naming pattern is `funcname_typeindex_globalindex` — e.g. `linear_3_42` is the 3rd distinct linear-layer position, 42nd op in the trace.

In [ ]:
# First 12 layers (input, embedding, attention prep)
print("First 12 layers:")
for layer in list(log.layers)[:12]:
    print(f"  {layer.layer_label}")

print()
print("Sample of mid-network layers (50-60):")
for layer in list(log.layers)[50:60]:
    print(f"  {layer.layer_label}")

Look at the **distinct functions** captured across the trace — TorchLens groups ops by their torch function family:

In [ ]:
from collections import Counter

func_counts = Counter()
for op in log.ops:
    # Drop the numeric suffix to get the function family
    family = str(op.layer_label).rsplit("_", 2)[0]
    func_counts[family] += 1

for func, n in func_counts.most_common(15):
    print(f"  {func:30s} x{n}")

## 5. Browse the modules

`log.modules` exposes every `nn.Module` instance in the source model — both called and uncalled. Each `Module` record carries class info, source location, the calls it participated in, and parameter/buffer ownership.

In [ ]:
# Top-level structure
print("Top-level modules (first 10):")
for mod in list(log.modules)[:10]:
    print(f"  {mod.address:50s}  ({mod.class_name})")

print()
print(f"Total modules: {log.num_modules}")
print(f"  with at least one call: {sum(1 for m in log.modules if m.num_calls > 0)}")

Find every attention module by class name:

In [ ]:
attn_modules = [m for m in log.modules if "attention" in m.class_name.lower()]
print(f"Found {len(attn_modules)} attention modules:")
for m in attn_modules:
    print(f"  {m.address:50s}  {m.class_name}")

## 6. Grab activations

There are several ways to pull a saved activation tensor out of the trace.

### 6a. By layer label

The simplest: index `log[layer_label]` returns the Layer record, and `.out` gives the saved output tensor.

In [ ]:
# Pick a specific layer mid-network
layer = log.layers[20]
print(f"layer label:  {layer.layer_label}")
print(f"output shape: {tuple(layer.out.shape)}")
print(f"output dtype: {layer.out.dtype}")
print(f"memory:       {layer.memory_str}")

### 6b. By module path

For a richer query — "what did the first transformer layer's attention block output?" — go through `log.modules[address]`. A `Module` aggregates all calls of that submodule; for single-call modules, `module.out` is a passthrough. For multi-call modules (RNNs, repeated layers), use `module.calls[N].out`.

In [ ]:
# DistilBERT has 6 transformer layers; grab the first attention block's output
attn0 = log.modules["transformer.layer.0.attention"]

print(f"module:        {attn0.address}")
print(f"class:         {attn0.class_name}")
print(f"num_calls:     {attn0.num_calls}")
print(f"num internal layers: {attn0.num_layers}")
print()

# Single call -> passthrough .out works
mc = attn0.calls[0]
out = mc.out
print(f"attention output shape: {tuple(out.shape)}")  # (batch, seq, hidden)

In [ ]:
# Compare attention outputs across layers
print("Attention outputs across the 6 transformer layers:")
for i in range(6):
    attn = log.modules[f"transformer.layer.{i}.attention"]
    out = attn.calls[0].out
    print(
        f"  layer {i}: shape={tuple(out.shape)}, mean={out.mean().item():+.4f}, std={out.std().item():.4f}"
    )

### 6c. By function family

To collect every op of a given function type — say, every layer normalization — filter `log.ops` or use the layer-label prefix:

In [ ]:
layernorm_layers = [layer for layer in log.layers if layer.layer_label.startswith("layernorm")]
print(f"Found {len(layernorm_layers)} layernorm layers:")
for layer in layernorm_layers[:8]:
    out = layer.out
    print(f"  {layer.layer_label:30s}  shape={tuple(out.shape)}")

### 6d. Drilling deeper — peek inside one attention block

Each attention module has its own internal layer labels you can list via `module.layer_labels`:

In [ ]:
attn0 = log.modules["transformer.layer.0.attention"]
print(f"Internal layers of {attn0.address}:")
for lbl in attn0.layer_labels:
    layer = log[lbl]
    out = layer.out
    shape_str = f"shape={tuple(out.shape)}" if hasattr(out, "shape") else f"({type(out).__name__})"
    print(f"  {lbl:30s}  {shape_str}")

## 7. Visualize the full graph

`log.draw(vis_outpath=...)` renders a Graphviz PDF (or SVG / PNG via `vis_renderer`). For models with many layers, the default rolled-loop view collapses repeated structures; setting `vis_mode='unrolled'` shows every pass explicitly.

In [ ]:
full_path = OUT / "distilbert_full"
log.draw(vis_outpath=str(full_path))
pdf_path = full_path.with_suffix(".pdf")
print(f"Rendered: {pdf_path}  ({pdf_path.stat().st_size / 1024:.1f} KB)")
display(IFrame(src=str(pdf_path), width="100%", height="600"))

## 8. Module-focused subgraph

Pass `module=<address>` to scope the visualization to just that submodule's subgraph. Great for zooming into a specific attention block or feedforward layer:

In [ ]:
attn_path = OUT / "distilbert_attn0"
log.draw(vis_outpath=str(attn_path), module="transformer.layer.0.attention")
pdf_path = attn_path.with_suffix(".pdf")
print(f"Rendered: {pdf_path}  ({pdf_path.stat().st_size / 1024:.1f} KB)")
display(IFrame(src=str(pdf_path), width="100%", height="500"))

## 9. Other architectures — quick sanity check

The HF bridge isn't BERT-specific. Same pattern works for autoregressive (GPT-2), seq2seq (T5), vision (ViT), and audio (Wav2Vec2) models. Quick example with GPT-2 small:

In [ ]:
gpt2 = AutoModel.from_pretrained("gpt2")
gpt2.eval()
gpt2_log = tl.bridge.hf.trace_text(gpt2, "TorchLens traces any eager PyTorch model.")
print(f"GPT-2 small: {gpt2_log.num_ops} ops, {gpt2_log.num_modules} modules")

# Different architecture, same query surface
attn0_gpt2 = gpt2_log.modules["h.0.attn"]
print(f"\nGPT-2 first attention block: {attn0_gpt2.class_name}")
print(f"  internal layers: {attn0_gpt2.num_layers}")
print(f"  output shape:    {tuple(attn0_gpt2.calls[0].out.shape)}")

## 10. Facets — semantic accessors on attention, norms, MLPs

TorchLens ships a **Facets framework**: a registry of "recipes" that produce derived semantic views on `Op` and `Module` records. Built-in recipes ship for common HF + torch families (attention blocks across 7+ class flavors, LayerNorm / RMSNorm, MLPs, Embedding). Users can register their own with `@tl.facets.register(...)`.

Both `Op` and `Module` carry a `.facets` field. It's dict-like AND supports attribute access. Lazy + cached: `view.keys()` is cheap, `view.q` triggers the recipe on first access and caches.

### What's registered

In [ ]:
# All recipes shipped with TorchLens
recipes = tl.facets.list()
print(f"{len(recipes)} built-in recipes:\n")
for r in recipes:
    classes = ", ".join(r.class_names) if r.class_names else "(predicate-only)"
    print(f"  {r.recipe_name:30s}  -- {classes}")

### Q, K, V access on an attention block

Module-level facets: for any recognized attention block, the recipe reshapes `q_lin`/`k_lin`/`v_lin` outputs to `(batch, seq, n_heads, d_head)` automatically.

In [ ]:
attn0 = log.modules["transformer.layer.0.attention"]

print(f"class:         {attn0.class_name}")
print(f"facet keys:    {sorted(attn0.facets.keys())}")
print(f"recipe source: {attn0.facets.recipe_source}")
print()
print(f"q shape:       {tuple(attn0.facets.q.shape)}    (B, S, n_heads, d_head)")
print(f"k shape:       {tuple(attn0.facets.k.shape)}")
print(f"v shape:       {tuple(attn0.facets.v.shape)}")
print(f"n_heads:       {attn0.facets.n_heads}")
print(f"d_head:        {attn0.facets.d_head}")
print(f"attn_out:      {tuple(attn0.facets.attn_out.shape)}")

### Per-head sub-view via `.head(i)`

Attention recipes expose a `.head(i)` method returning a sub-view scoped to one head — `view.head(3).q` is equivalent to `view.q[:, :, 3, :]` but reads more naturally.

In [ ]:
h = attn0.facets.head(3)
print(f"head(3).q shape: {tuple(h.q.shape)}    (B, S, d_head)")
print(f"head(3).k shape: {tuple(h.k.shape)}")
print(f"head(3).v shape: {tuple(h.v.shape)}")

# Equivalent direct indexing

assert torch.allclose(h.q, attn0.facets.q[:, :, 3, :])
print("\nEquivalent to direct indexing — confirmed.")

### Sweeping across all attention blocks

`log.attention_blocks()` returns every module matched by an attention recipe. Useful for cross-layer analysis.

In [ ]:
print("Q-norm of each attention block:")
for blk in log.attention_blocks():
    q = blk.facets.q
    q_norm = q.norm(dim=-1).mean().item()
    print(f"  {blk.address:50s} mean |q| = {q_norm:.3f}")

### Generic `modules_with_facet(name)` finder

For non-attention queries — e.g., grab every LayerNorm's output across the model.

In [ ]:
print("LayerNorm modules — shapes of normalized output and (if available) gamma:")
for ln in log.modules_with_facet("normalized"):
    out = ln.facets.normalized
    out_shape = tuple(out.shape) if out is not None and hasattr(out, "shape") else None
    # gamma may not populate in default capture mode (param values not eagerly loaded);
    # use .has() to test, since declared keys aren't guaranteed to materialize at compute time
    gamma = ln.facets.get("gamma") if hasattr(ln.facets, "get") else None
    gamma_repr = tuple(gamma.shape) if gamma is not None and hasattr(gamma, "shape") else gamma
    print(f"  {ln.address:45s}  output={out_shape}  gamma={gamma_repr}")

### MLP facets

For MLP blocks, the recipe exposes intermediate and output activations.

In [ ]:
# Find an MLP block — DistilBERT uses ffn
ffn0 = log.modules["transformer.layer.0.ffn"]
print(f"class:        {ffn0.class_name}")
print(f"facet keys:   {sorted(ffn0.facets.keys())}")
print()
for k in ffn0.facets.keys():
    v = ffn0.facets[k]
    shape = tuple(v.shape) if hasattr(v, "shape") else v
    print(f"  {k:15s} -> {shape}")

### The fused-SDPA limitation

PyTorch's fused scaled-dot-product-attention kernel doesn't expose the post-softmax attention pattern — it lives inside C++ code. TorchLens detects this and raises an informative error rather than silently returning `None`:

In [ ]:
try:
    attn0.facets.pattern
except RuntimeError as e:
    print(f"RuntimeError: {e}")

To get the attention pattern back, re-load the model with `_attn_implementation='eager'`:

```python
model = AutoModel.from_pretrained('distilbert-base-uncased',
                                   attn_implementation='eager')
```

Then the recipe can recover the softmax output as a separate Op.

### Register your own recipe

Recipes are user-extensible. Register a function against a class name (or qualname, or arbitrary predicate) and `.facets` on every matching module gains your additions.

In [ ]:
# Register a custom facet on DistilBert attention blocks computing per-head
# norm of the attention output. IMPORTANT: don't access `mod.facets.q` from inside
# a recipe -- that recurses through the same view. Compute from the module's own
# captured calls / activations instead.
@tl.facets.register(class_name="DistilBertSdpaAttention", facets=("out_l2_per_token",))
def my_out_norm_recipe(mod):
    out = mod.calls[0].out  # the attention block's output tensor (B, S, hidden)
    if out is None or not hasattr(out, "norm"):
        return {}
    return {"out_l2_per_token": out.norm(dim=-1)}  # (B, S) -- L2 norm per token


# Re-create FacetView to pick up the newly registered recipe
del attn0.facets  # drops cached FacetView; next access re-matches recipes
print(f"After registration, recipe_source: {attn0.facets.recipe_source}")
print(f"out_l2_per_token shape: {tuple(attn0.facets.out_l2_per_token.shape)}    (B, S)")
print(f"mean over tokens:  {attn0.facets.out_l2_per_token.mean().item():.4f}")

### Discovery helpers

```python
tl.facets.list()                          # all registered recipes
tl.facets.list(scope='module')            # only Module-scoped
tl.facets.list(class_name='*Attention')   # glob match
tl.facets.info('DistilBertSdpaAttention') # what facets a class gets
```

In [ ]:
# What recipes match DistilBertSdpaAttention?
info = tl.facets.info("DistilBertSdpaAttention")
print(f"Recipes:  {info['recipes']}")
print(f"Facets:   {sorted(info['facets'])}")

## What to explore next

- **Forward + backward**: pass `backward_ready=True` to `trace_text` (or `tl.trace`), then call `log.backward(loss)` to capture gradients and grad-fn nodes.
- **Selective capture**: `layers_to_save=[...]` to keep only specific layers' activations in memory — useful for large models where you only care about a few sites.
- **Intervention**: `tl.trace(..., intervention_ready=True)` enables `log.fork(name)` + `attach_hooks(selector, fn)` + `rerun(model, x)` for ablation / patching experiments.
- **Compare two traces**: wrap two traces in a `Bundle` and use `Super[Trace]` accessors to align them across the architecture.
- **Save the trace portably**: `log.save('mytrace.tlspec')` writes a portable artifact; `tl.load('mytrace.tlspec')` reloads it without needing the model.
- **HF Hub**: `tl.bridge.huggingface.push_to_hub(log, repo_id='...')` uploads a trace to share with collaborators.